# Practicing Programmatic Prompting for Agents

## Building a Quasi-Agent

For practice, we are going to write a quasi-agent that can write Python functions based on user requirements. It isn't quite a real agent, it can't react and adapt, but it can do something useful for us.

The quasi-agent will ask the user what they want code for, write the code for the function, add documentation, and finally include test cases using the `unittest` framework. This exercise will help you understand how to maintain context across multiple prompts and manage the information flow between the user and the LLM. It will also help you understand the pain of trying to parse and handle the output of an LLM that is not always consistent.

## Reminder

Before you start, make sure you have the LiteLLM library installed:

```bash
pip install litellm
```

Make sure that you have an API key for the LLM provider you plan to use. You can read about how to get an API key from OpenAI [here](https://platform.openai.com/docs/quickstart).

Make sure that you set this key as an environment variable in your terminal before running the code. See the [LiteLLM documentation](https://docs.litellm.ai/) for more information.

## Practice Exercise

This exercise will allow you to practice programmatically sending prompts to an LLM and managing memory.

For this exercise, you should write a program that uses sequential prompts to generate any Python function based on user input. The program should:

### First Prompt:
- Ask the user what function they want to create
- Ask the LLM to write a basic Python function based on the user's description
- Store the response for use in subsequent prompts
- Parse the response to separate the code from the commentary by the LLM

### Second Prompt:
- Pass the code generated from the first prompt
- Ask the LLM to add comprehensive documentation including:
  - Function description
  - Parameter descriptions
  - Return value description
  - Example usage
  - Edge cases

### Third Prompt:
- Pass the documented code generated from the second prompt
- Ask the LLM to add test cases using Python's `unittest` framework
- Tests should cover:
  - Basic functionality
  - Edge cases
  - Error cases
  - Various input scenarios

### Requirements:
- Use the LiteLLM library
- Maintain conversation context between prompts
- Print each step of the development process
- Save the final version to a Python file

> **Bonus:** If you want to practice further, try using the system message to force the LLM to always output code that has a specific style or uses particular libraries.
```

# Practicing Programmatic Prompting for Agents (Solution)

Take a minute to scroll to the bottom to look at the complete solution. Once you are done, come back here and we will walk through the solution.

## Understanding the Architecture

Our quasi-agent works through three key steps:

1. Initial code generation
2. Documentation enhancement
3. Test case creation

The magic happens in how we maintain context between these steps, ensuring each builds on the previous results.

## Core Components

Let's break down the key pieces:
```python
def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get response"""
    response = completion(
        model="openai/gpt-4",
        messages=messages,
        max_tokens=1024
    )
    return response.choices[0].message.content
```


This function handles our LLM interactions using ChatML format. Each message includes a role (“system”, “user”, or “assistant”) and content.
```python
def extract_code_block(response: str) -> str:
    """Extract code block from response"""
    if not '```' in response:
        return response

    code_block = response.split('```')[1].strip()
    if code_block.startswith("python"):
        code_block = code_block[6:]

    return code_block
```

The LLM often includes commentary with its code. This function extracts just the code block, making it easier to build upon in subsequent prompts.

## The Development Process

The main function, `develop_custom_function()`, orchestrates three phases of development:

### Phase 1: Initial Code Generation

```python
messages = [
    {"role": "system", "content": "You are a Python expert helping to develop a function."}
]

messages.append({
    "role": "user",
    "content": f"Write a Python function that {function_description}. Output the function in a ```python code block```."
})
```

We start with a system message establishing the LLM's role, then request initial code based on the user's description.

### Phase 2: Documentation Enhancement

```python
messages.append({
    "role": "assistant", 
    "content": "```python\n\n"+initial_function+"\n\n```"
})

messages.append({
    "role": "user",
    "content": "Add comprehensive documentation to this function..."
})
```

Notice how we feed back the code but strip any commentary. This keeps the LLM focused on just the code structure.

### Phase 3: Test Case Generation

```python
messages.append({
    "role": "assistant", 
    "content": "```python\n\n"+documented_function+"\n\n```"
})

messages.append({
    "role": "user",
    "content": "Add unittest test cases for this function..."
})
```

Again, we maintain clean context by showing only the documented code.

## Memory Management Through Message History

The key insight is how we manage "memory" through the `messages` list. Each step builds on previous responses, but we carefully control what the LLM sees:

- We only show the code, not the commentary
- Each message provides specific instruction for the next enhancement
- The context builds progressively through the message history

For example, when adding documentation, the LLM sees:
- It's a Python expert (system message)
- The original code (previous response)
- The request for documentation (current task)

This focused context helps ensure consistent, high-quality output.

## Usage Example

Here's how it works in practice:

```
>>> function_code, tests, filename = develop_custom_function()
What kind of function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: Calculate fibonacci sequence up to n

=== Initial Function ===
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    sequence = [0, 1]
    while len(sequence) < n:
        sequence.append(sequence[-1] + sequence[-2])
    return sequence

=== Documented Function ===
[... function with added documentation ...]

=== Test Cases ===
[... unittest test cases ...]

Final code has been saved to calculate_fibonacci_sequence_up.py
```

## Learning from This Design

This quasi-agent teaches us several important lessons about building LLM-powered systems:

1. **Prompt Chaining:** Breaking complex tasks into sequential steps makes them more manageable.

2. **Context Management:** Carefully controlling what the LLM sees helps maintain focus and consistency.

3. **Output Processing:** Having robust ways to extract and clean LLM output is crucial.

4. **Progressive Enhancement:** Building features iteratively (code → docs → tests) creates better results than trying to do everything at once.

These principles apply even when building more complex, fully agentic systems.
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = api_key

In [2]:
from litellm import completion
from typing import List, Dict

def generate_response(messages: List[Dict]) -> str:
    """Call Groq LLM to get response"""
    response = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=messages,
        max_tokens=1024
    )
    return response.choices[0].message.content

In [3]:
def extract_code_block(response: str) -> str:
   """Extract code block from response"""

   if not '```' in response:
      return response

   code_block = response.split('```')[1].strip()
   # Check for "python" at the start and remove

   if code_block.startswith("python"):
      code_block = code_block[6:]

   return code_block

In [4]:
def develop_custom_function():
   # Get user input for function description
   print("\nWhat kind of function would you like to create?")
   print("Example: 'A function that calculates the factorial of a number'")
   print("Your description: ", end='')
   function_description = input().strip()

   # Initialize conversation with system prompt
   messages = [
      {"role": "system", "content": "You are a Python expert helping to develop a function."}
   ]

   # First prompt - Basic function
   messages.append({
      "role": "user",
      "content": f"Write a Python function that {function_description}. Output the function in a ```python code block```."
   })
   initial_function = generate_response(messages)

   # Parse the response to get the function code
   initial_function = extract_code_block(initial_function)

   print("\n=== Initial Function ===")
   print(initial_function)

   # Add assistant's response to conversation
   # Notice that I am purposely causing it to forget its commentary and just see the code so that
   # it appears that is always outputting just code.
   messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})

   # Second prompt - Add documentation
   messages.append({
      "role": "user",
      "content": "Add comprehensive documentation to this function, including description, parameters, "
                 "return value, examples, and edge cases. Output the function in a ```python code block```."
   })
   documented_function = generate_response(messages)
   documented_function = extract_code_block(documented_function)
   print("\n=== Documented Function ===")
   print(documented_function)

   # Add documentation response to conversation
   messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+documented_function+"\n\n\`\`\`"})

   # Third prompt - Add test cases
   messages.append({
      "role": "user",
      "content": "Add unittest test cases for this function, including tests for basic functionality, "
                 "edge cases, error cases, and various input scenarios. Output the code in a \`\`\`python code block\`\`\`."
   })
   test_cases = generate_response(messages)
   # We will likely run into random problems here depending on if it outputs JUST the test cases or the
   # test cases AND the code. This is the type of issue we will learn to work through with agents in the course.
   test_cases = extract_code_block(test_cases)
   print("\n=== Test Cases ===")
   print(test_cases)

   # Generate filename from function description
   filename = function_description.lower()
   filename = ''.join(c for c in filename if c.isalnum() or c.isspace())
   filename = filename.replace(' ', '_')[:30] + '.py'

   # Save final version
   with open(filename, 'w') as f:
      f.write(documented_function + '\n\n' + test_cases)

   return documented_function, test_cases, filename

<>:29: SyntaxWarning: invalid escape sequence '\`'
<>:29: SyntaxWarning: invalid escape sequence '\`'
<>:43: SyntaxWarning: invalid escape sequence '\`'
<>:43: SyntaxWarning: invalid escape sequence '\`'
<>:49: SyntaxWarning: invalid escape sequence '\`'
<>:29: SyntaxWarning: invalid escape sequence '\`'
<>:29: SyntaxWarning: invalid escape sequence '\`'
<>:43: SyntaxWarning: invalid escape sequence '\`'
<>:43: SyntaxWarning: invalid escape sequence '\`'
<>:49: SyntaxWarning: invalid escape sequence '\`'
/tmp/ipykernel_43345/27984788.py:29: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})
/tmp/ipykernel_43345/27984788.py:29: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "content": "\`\`\`python\n\n"+initial_function+"\n\n\`\`\`"})
/tmp/ipykernel_43345/27984788.py:43: SyntaxWarning: invalid escape sequence '\`'
  messages.append({"role": "assistant", "co

In [5]:
if __name__ == "__main__":


   function_code, tests, filename = develop_custom_function()
   print(f"\nFinal code has been saved to {filename}")


What kind of function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: 
=== Initial Function ===

import math

def calculate_square_root():
    number = 100
    square_root = math.sqrt(number)
    return square_root

result = calculate_square_root()
print("The square root of 100 is:", result)

=== Documented Function ===

"""
Module for calculating the square root of a number.

This module provides a function to calculate the square root of a given number.
It uses the math library in Python to achieve this.
"""

import math

def calculate_square_root(number: float = 100) -> float:
    """
    Calculate the square root of a given number.

    Args:
    number (float): The number for which the square root is to be calculated. Defaults to 100.

    Returns:
    float: The square root of the given number.

    Examples:
    >>> calculate_square_root(100)
    10.0
    >>> calculate_square_root(16)
    4.0

    Edge Cases:
    >>> c